In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import os
import pandas as pd
from dotenv import load_dotenv
from mal_client import MALClient
from anime_data import AnimeDataClient
from anime_recommender import SimilarityRecommender

load_dotenv(PROJECT_ROOT / ".env")

client_id = os.getenv("CLIENT_ID")

In [ ]:
anime_data_client = AnimeDataClient(client_id, cache_file=PROJECT_ROOT / "data" / "anime_cache.json")

In [4]:
anime_data = anime_data_client.get_cache()

Finding best numerical feature set

In [5]:
# import numpy as np
# import pandas as pd
# from sklearn.linear_model import LinearRegression

# numeric_candidates = pd.DataFrame(anime_data.values())

# statistics_df = pd.json_normalize(numeric_candidates["statistics"])
# statistics_df = statistics_df.rename(columns={
#     "num_list_users": "statistics_num_list_users",
#     "status.watching": "watching",
#     "status.completed": "completed",
#     "status.on_hold": "on_hold",
#     "status.dropped": "dropped",
#     "status.plan_to_watch": "plan_to_watch",
# })
# statistics_df = statistics_df.apply(pd.to_numeric, errors="coerce")

# numeric_candidates = pd.concat(
#     [numeric_candidates.drop(columns=["statistics"]), statistics_df],
#     axis=1,
# )

# numeric_columns = [
#     "mean",
#     "rank",
#     "popularity",
#     "num_list_users",
#     "num_scoring_users",
#     "num_episodes",
#     "statistics_num_list_users",
#     "watching",
#     "completed",
#     "on_hold",
#     "dropped",
#     "plan_to_watch",
# ]

# numeric_analysis_df = (
#     numeric_candidates[numeric_columns]
#     .apply(pd.to_numeric, errors="coerce")
#     .dropna()
# )

# numeric_analysis_df.head()

In [6]:
# def calculate_vif(df):
#     rows = []
#     for target_col in df.columns:
#         X = df.drop(columns=[target_col]).to_numpy(dtype=float)
#         y = df[target_col].to_numpy(dtype=float)

#         model = LinearRegression()
#         model.fit(X, y)
#         r_squared = model.score(X, y)

#         vif = np.inf if np.isclose(1 - r_squared, 0) else 1 / (1 - r_squared)
#         rows.append({
#             "feature": target_col,
#             "r_squared_from_other_features": r_squared,
#             "vif": vif,
#         })

#     return pd.DataFrame(rows).sort_values("vif", ascending=False)

# reduced_numeric_analysis_df = numeric_analysis_df.drop(
#     columns=[
#         "completed",
#         "on_hold",
#         "statistics_num_list_users",
#         # "watching",
#         "dropped",
#         "plan_to_watch",
#         "num_list_users",
#         "num_scoring_users",
#         "num_episodes",
#         "rank",
#         # "popularity",
#         # "mean",
#     ],
#     errors="ignore",
# )

# vif_results = calculate_vif(reduced_numeric_analysis_df)
# vif_results

Build features

In [7]:
from anime_features import AnimeFeatureBuilder

builder = AnimeFeatureBuilder(
    anime_data,
    max_tfidf_features=4000,
    n_svd_components=400
)

anime_df_num = builder.build_num_features().set_index("id")
anime_genres_df = builder.build_genre_features().set_index("anime_id")
anime_studios_df = builder.build_studio_features().set_index("anime_id")
synopsis_tfidf, synopsis_features = builder.build_synopsis_features()
synopsis_svd_df = builder.apply_svd(synopsis_tfidf, synopsis_features)

anime_df_complete = pd.concat([
    anime_df_num,
    anime_genres_df,
    synopsis_svd_df,
], axis=1).dropna()

builder.svd_explained_variance

np.float64(0.40687277196386595)

In [8]:
anime_df_num_new = pd.DataFrame(anime_data.values())
anime_df_num_new = anime_df_num_new.drop(
    columns=[
        "main_picture",
        "title",
        "synopsis",
        "media_type",
        "status",
        "genres",
        "rating",
        "recommendations",
        "studios",
        "related_anime",
    ],
    errors="ignore",
)

statistics_df = pd.json_normalize(anime_df_num_new["statistics"])
statistics_df = statistics_df.rename(columns={
    "num_list_users": "statistics_num_list_users",
    "status.watching": "watching",
    "status.completed": "completed",
    "status.on_hold": "on_hold",
    "status.dropped": "dropped",
    "status.plan_to_watch": "plan_to_watch",
})
statistics_df = statistics_df.apply(pd.to_numeric, errors="coerce")

anime_df_num_new = pd.concat([anime_df_num_new.drop(columns=["statistics"]), statistics_df], axis=1)

anime_df_num_new = anime_df_num_new.drop(
    columns=[
        "completed",
        "on_hold",
        "statistics_num_list_users",
        # "watching",
        "dropped",
        "plan_to_watch",
        "num_list_users",
        "num_scoring_users",
        "num_episodes",
        "rank",
        # "popularity",
        # "mean",
    ],
    errors="ignore",).set_index("id").dropna()

anime_df_num_new

,mean,popularity,watching
id,,,
19,8.89,115,180331
1827,8.12,1206,14229
5114,9.11,3,289840
11061,9.03,8,385809
13125,8.24,276,56382
...,...,...,...
1705,5.16,11753,176
30782,5.14,2393,6726
30947,5.17,11102,232


In [9]:
# Pick the feature set to use below.
# anime_df = anime_df_complete
anime_df = pd.concat([anime_df_num_new, anime_genres_df, synopsis_svd_df], axis=1).dropna()
# anime_df = pd.concat([anime_df_complete, anime_studios_df], axis=1).dropna()
# anime_df = anime_df_num.dropna()
# anime_df = anime_genres_df.dropna()
# anime_df = anime_studios_df.dropna()
# anime_df = pd.concat([anime_df_num, anime_genres_df], axis=1).dropna()
# anime_df = pd.concat([anime_df_num, anime_studios_df], axis=1).dropna()
# anime_df = pd.concat([anime_df_num, synopsis_svd_df], axis=1).dropna()
# anime_df = pd.concat([anime_genres_df, synopsis_svd_df], axis=1).dropna()
# anime_df = pd.concat([anime_genres_df, anime_studios_df], axis=1).dropna()
# anime_df = pd.concat([anime_df_num, anime_genres_df, anime_studios_df], axis=1).dropna()
# anime_df = pd.concat([anime_df_num, synopsis_svd_df, anime_studios_df], axis=1).dropna()

Convert each anime in df to vectors

In [10]:
recommender = SimilarityRecommender()
anime_vectors = recommender.create_anime_vectors(anime_df)
anime_df_scaled = recommender.anime_df_scaled

anime_df_scaled.head()

,mean,popularity,watching,Action,Adult Cast,Adventure,Anthropomorphic,Avant Garde,Award Winning,Boys Love,...,synopsis_svd_390,synopsis_svd_391,synopsis_svd_392,synopsis_svd_393,synopsis_svd_394,synopsis_svd_395,synopsis_svd_396,synopsis_svd_397,synopsis_svd_398,synopsis_svd_399
19,2.541591,-1.087001,4.171334,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.034993,0.000500,-0.013383,-0.003008,-0.015781,-0.021830,0.006703,-0.008218,0.029499,-0.000379
1827,1.523591,-0.858080,0.004501,1.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.023757,-0.012438,-0.009539,-0.022266,0.015162,0.031193,-0.014519,-0.017107,-0.025170,-0.007054
5114,2.832448,-1.110502,6.918476,1.0,0.0,1.0,0.0,0.0,0.0,0.0,...,-0.019361,-0.003685,0.030090,-0.048054,-0.020457,-0.028526,-0.016868,0.007993,-0.035077,0.051686
11061,2.726682,-1.109453,9.325954,1.0,0.0,1.0,0.0,0.0,0.0,0.0,...,-0.021416,-0.034208,-0.002527,0.011559,-0.019431,0.013987,-0.012849,-0.003094,0.020161,-0.021612
13125,1.682241,-1.053219,1.061950,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.001195,0.043650,0.021355,0.002969,-0.000660,0.017009,0.039973,-0.006862,0.017056,-0.022117


Get user Data

In [11]:
username = "chekkit"
user_client = MALClient(client_id)

user_data = user_client.get_user_data(username)
user_scores = user_client.get_scores(user_data)

Hit Rate

In [12]:
# from anime_evaluation import HitRateEvaluator, RankingMetricEvaluator

# n_runs = 500
# result_top_ks = (5, 10)
# uncertainty_weight = 8.5

# evaluator = HitRateEvaluator(
#     anime_df_scaled=anime_df_scaled,
#     anime_df=anime_df,
#     scores=user_scores,
#     heldout_fraction=0.25,
# )

# (
#     bayesian_results,
#     bayesian_summary,
#     best_bayesian_weights,
#     baseline_results,
#     baseline_summary,
# ) = evaluator.tune_bayesian_uncertainty(
#     weights=[uncertainty_weight],
#     n_runs=n_runs,
#     top_ks=result_top_ks,
#     random_state=42,
# )

# ranking_evaluator = RankingMetricEvaluator(
#     anime_df_scaled=anime_df_scaled,
#     anime_df=anime_df,
#     scores=user_scores,
#     heldout_fraction=0.25,
# )

# ranking_results, ranking_summary = ranking_evaluator.evaluate_bayesian(
#     uncertainty_weight=uncertainty_weight,
#     n_runs=n_runs,
#     top_ks=result_top_ks,
#     random_state=42,
#     include_global_mean=True,
# )

# bayesian_ranking_summary = ranking_summary[
#     ranking_summary["model"] == "bayesian_ridge"
# ][[
#     "k",
#     "avg_ndcg_at_k",
#     "std_ndcg_at_k",
#     "avg_mrr_at_k",
#     "std_mrr_at_k",
#     "avg_relevant_hits_at_k",
#     "avg_strong_hits_at_k",
# ]]

# if baseline_summary is not None and not baseline_summary.empty:
#     average_metrics = bayesian_summary.merge(
#         baseline_summary,
#         on="k",
#         how="left",
#     )
# else:
#     average_metrics = bayesian_summary.copy()

# average_metrics = average_metrics.merge(
#     bayesian_ranking_summary,
#     on="k",
#     how="left",
# )

# average_metrics = average_metrics.rename(
#     columns={"uncertainty_weight": "bayesian_uncertainty_weight"}
# )

# average_metrics

## Numeric Combo Sweep

This sweep keeps the core numeric signals (`mean`, `popularity`, `num_episodes`) and tests all 64 combinations of optional activity/support signals inside the full feature context: numeric combo + genres + synopsis SVD. Precision@K is the primary metric, and NDCG@K is included as a tie-breaker when precision is close. Use a smaller run count for the sweep, then rerun the top few combos with more runs.

In [14]:
from itertools import combinations
from anime_evaluation import HitRateEvaluator, RankingMetricEvaluator
from anime_recommender import SimilarityRecommender

core_numeric_features = ["mean", "popularity", "statistics_num_list_users"]
optional_numeric_features = [
    "watching",
    "completed",
    "on_hold",
    "dropped",
    "plan_to_watch",
]

raw_numeric_candidates = pd.DataFrame(anime_data.values())
statistics_df = pd.json_normalize(raw_numeric_candidates["statistics"])
statistics_df = statistics_df.rename(columns={
    "num_list_users": "statistics_num_list_users",
    "status.watching": "watching",
    "status.completed": "completed",
    "status.on_hold": "on_hold",
    "status.dropped": "dropped",
    "status.plan_to_watch": "plan_to_watch",
})

numeric_candidates = pd.concat(
    [
        raw_numeric_candidates[["id", "mean", "popularity"]],
        statistics_df[
            [
                "statistics_num_list_users",
                "watching",
                "completed",
                "on_hold",
                "dropped",
                "plan_to_watch",
            ]
        ],
    ],
    axis=1,
)

available_core_numeric_features = [
    column for column in core_numeric_features if column in numeric_candidates.columns
]
available_optional_numeric_features = [
    column for column in optional_numeric_features if column in numeric_candidates.columns
]
numeric_columns = available_core_numeric_features + available_optional_numeric_features
missing_numeric_columns = sorted(
    set(core_numeric_features + optional_numeric_features) - set(numeric_columns)
)
if missing_numeric_columns:
    print(f"Skipping unavailable numeric columns: {missing_numeric_columns}")

numeric_combo_base = numeric_candidates[["id"] + numeric_columns].copy()
for column in numeric_columns:
    numeric_combo_base[column] = pd.to_numeric(
        numeric_combo_base[column],
        errors="coerce",
    )
numeric_combo_base = numeric_combo_base.set_index("id").dropna()

combo_n_runs = 20
combo_top_ks = (5, 10)
combo_uncertainty_weight = 14
combo_clip_predictions = False
combo_rows = []

for combo_size in range(len(available_optional_numeric_features) + 1):
    for optional_combo in combinations(available_optional_numeric_features, combo_size):
        selected_numeric_features = available_core_numeric_features + list(optional_combo)
        feature_set_name = "core"
        if optional_combo:
            feature_set_name += "_" + "_".join(optional_combo)

        combo_anime_df = pd.concat(
            [
                numeric_combo_base[selected_numeric_features],
                anime_genres_df,
                synopsis_svd_df,
            ],
            axis=1,
        ).dropna()

        combo_recommender = SimilarityRecommender()
        combo_recommender.create_anime_vectors(combo_anime_df)

        combo_evaluator = HitRateEvaluator(
            anime_df_scaled=combo_recommender.anime_df_scaled,
            anime_df=combo_anime_df,
            scores=user_scores,
            heldout_fraction=0.25,
        )

        _, combo_summary, _, _, combo_baseline_summary = (
            combo_evaluator.tune_bayesian_uncertainty(
                weights=[combo_uncertainty_weight],
                n_runs=combo_n_runs,
                top_ks=combo_top_ks,
                random_state=42,
                clip_predictions=combo_clip_predictions,
            )
        )

        combo_ranking_evaluator = RankingMetricEvaluator(
            anime_df_scaled=combo_recommender.anime_df_scaled,
            anime_df=combo_anime_df,
            scores=user_scores,
            heldout_fraction=0.25,
        )
        _, combo_ranking_summary = combo_ranking_evaluator.evaluate_bayesian(
            uncertainty_weight=combo_uncertainty_weight,
            n_runs=combo_n_runs,
            top_ks=combo_top_ks,
            random_state=42,
            include_global_mean=False,
            clip_predictions=combo_clip_predictions,
        )
        combo_ranking_summary = (
            combo_ranking_summary[
                combo_ranking_summary["model"] == "bayesian_ridge"
            ][[
                "k",
                "avg_ndcg_at_k",
                "std_ndcg_at_k",
                "avg_mrr_at_k",
                "avg_relevant_hits_at_k",
                "avg_strong_hits_at_k",
            ]]
        )
        combo_summary = combo_summary.merge(
            combo_ranking_summary,
            on="k",
            how="left",
        )

        if combo_baseline_summary is not None and not combo_baseline_summary.empty:
            combo_summary = combo_summary.merge(
                combo_baseline_summary,
                on="k",
                how="left",
            )

        for row in combo_summary.to_dict("records"):
            row["feature_set"] = feature_set_name
            row["numeric_features"] = ", ".join(selected_numeric_features)
            row["n_numeric_features"] = len(selected_numeric_features)
            row["n_runs"] = combo_n_runs
            row["clip_predictions"] = combo_clip_predictions
            combo_rows.append(row)

numeric_combo_results = pd.DataFrame(combo_rows)
numeric_combo_summary = numeric_combo_results.sort_values(
    ["k", "avg_precision_at_k", "avg_ndcg_at_k", "avg_hit_rate"],
    ascending=[True, False, False, False],
)

numeric_combo_summary.head(20)


,uncertainty_weight,k,avg_precision_at_k,std_precision_at_k,avg_hit_rate,std_hit_rate,avg_hits,baseline_avg_precision_at_k,baseline_std_precision_at_k,baseline_avg_hit_rate,baseline_std_hit_rate,baseline_avg_hits,feature_set,numeric_features,n_numeric_features,n_runs
4,7.5,5,0.64,0.178885,0.103226,0.028852,3.20,0.17,0.134164,0.027419,0.021639,0.85,core_watching,"mean, popularity, num_episodes, watching",4,20
10,7.5,5,0.63,0.217885,0.101613,0.035143,3.15,0.17,0.134164,0.027419,0.021639,0.85,core_dropped,"mean, popularity, num_episodes, dropped",4,20
8,7.5,5,0.62,0.214231,0.100000,0.034553,3.10,0.17,0.134164,0.027419,0.021639,0.85,core_on_hold,"mean, popularity, num_episodes, on_hold",4,20
18,7.5,5,0.60,0.215211,0.096774,0.034711,3.00,0.17,0.134164,0.027419,0.021639,0.85,core_num_scoring_users_on_hold,"mean, popularity, num_episodes, num_scoring_us...",5,20
34,7.5,5,0.60,0.224781,0.096774,0.036255,3.00,0.17,0.134164,0.027419,0.021639,0.85,core_completed_dropped,"mean, popularity, num_episodes, completed, dro...",5,20
38,7.5,5,0.60,0.215211,0.096774,0.034711,3.00,0.17,0.134164,0.027419,0.021639,0.85,core_on_hold_dropped,"mean, popularity, num_episodes, on_hold, dropped",5,20
20,7.5,5,0.59,0.219809,0.095161,0.035453,2.95,0.17,0.134164,0.027419,0.021639,0.85,core_num_scoring_users_dropped,"mean, popularity, num_episodes, num_scoring_us...",5,20
32,7.5,5,0.59,0.219809,0.095161,0.035453,2.95,0.17,0.134164,0.027419,0.021639,0.85,core_completed_on_hold,"mean, popularity, num_episodes, completed, on_...",5,20
42,7.5,5,0.59,0.210013,0.095161,0.033873,2.95,0.17,0.134164,0.027419,0.021639,0.85,core_dropped_plan_to_watch,"mean, popularity, num_episodes, dropped, plan_...",5,20
52,7.5,5,0.59,0.219809,0.095161,0.035453,2.95,0.17,0.134164,0.027419,0.021639,0.85,core_num_scoring_users_completed_on_hold,"mean, popularity, num_episodes, num_scoring_us...",6,20


## Numeric Combo Sweep

This broader sweep tests all non-empty combinations of the available builder-cleaned numeric candidate set inside the full feature context: numeric combo + genres + synopsis SVD. With all 8 requested numeric features available, this produces `2^8 - 1 = 255` feature sets. On the current cache, unavailable fields are skipped before the sweep runs.

In [13]:
from itertools import combinations
from anime_evaluation import HitRateEvaluator
from anime_recommender import SimilarityRecommender

numeric_sweep_features = [
    "mean",
    "popularity",
    "statistics_num_list_users",
    "watching",
    "completed",
    "on_hold",
    "dropped",
    "plan_to_watch",
]

raw_numeric_candidates = pd.DataFrame(anime_data.values())
statistics_df = pd.json_normalize(raw_numeric_candidates["statistics"])
statistics_df = statistics_df.rename(columns={
    "num_list_users": "statistics_num_list_users",
    "status.watching": "watching",
    "status.completed": "completed",
    "status.on_hold": "on_hold",
    "status.dropped": "dropped",
    "status.plan_to_watch": "plan_to_watch",
})

numeric_candidates = pd.concat(
    [
        raw_numeric_candidates[["id", "mean", "popularity"]],
        statistics_df[
            [
                "statistics_num_list_users",
                "watching",
                "completed",
                "on_hold",
                "dropped",
                "plan_to_watch",
            ]
        ],
    ],
    axis=1,
)

available_numeric_sweep_features = [
    column
    for column in numeric_sweep_features
    if column in numeric_candidates.columns
]
missing_numeric_sweep_features = sorted(
    set(numeric_sweep_features) - set(available_numeric_sweep_features)
)
if missing_numeric_sweep_features:
    print(f"Skipping unavailable numeric columns: {missing_numeric_sweep_features}")

numeric_combo_base = numeric_candidates[["id"] + available_numeric_sweep_features].copy()
for column in available_numeric_sweep_features:
    numeric_combo_base[column] = pd.to_numeric(
        numeric_combo_base[column],
        errors="coerce",
    )
numeric_combo_base = numeric_combo_base.set_index("id").dropna()

print(
    "Feature frame diagnostic: "
    f"numeric_rows={len(numeric_combo_base)}, "
    f"genre_rows={len(anime_genres_df)}, "
    f"synopsis_rows={len(synopsis_svd_df)}, "
    f"numeric_genre_overlap={len(numeric_combo_base.index.intersection(anime_genres_df.index))}, "
    f"numeric_synopsis_overlap={len(numeric_combo_base.index.intersection(synopsis_svd_df.index))}, "
    f"all_overlap={len(numeric_combo_base.index.intersection(anime_genres_df.index).intersection(synopsis_svd_df.index))}",
    flush=True,
)

diagnostic_features = [
    column
    for column in ["mean", "popularity", "watching"]
    if column in numeric_combo_base.columns
]
diagnostic_anime_df = pd.concat(
    [
        numeric_combo_base[diagnostic_features],
        anime_genres_df,
        synopsis_svd_df,
    ],
    axis=1,
).dropna()
diagnostic_recommender = SimilarityRecommender()
diagnostic_recommender.create_anime_vectors(diagnostic_anime_df)
diagnostic_evaluator = HitRateEvaluator(
    anime_df_scaled=diagnostic_recommender.anime_df_scaled,
    anime_df=diagnostic_anime_df,
    scores=user_scores,
    heldout_fraction=0.25,
)
print("User score counts:")
print(pd.Series(user_scores).value_counts().sort_index().to_string())
print(
    "Evaluator diagnostic: "
    f"user_scores={len(user_scores)}, "
    f"candidate_rows={len(diagnostic_anime_df)}, "
    f"rated_overlap={len(diagnostic_evaluator.rated_eval)}, "
    f"like_threshold={diagnostic_evaluator.like_threshold:.3f}, "
    f"liked_overlap={len(diagnostic_evaluator.liked_eval)}, "
    f"heldout_likes_per_split={len(diagnostic_evaluator.liked_eval.sample(frac=0.25, random_state=42))}",
    flush=True,
)

all_combo_n_runs = 20
all_combo_top_ks = (5, 10)
all_combo_uncertainty_weight = 14
all_combo_clip_predictions = False
all_combo_rows = []

n_combos = 2 ** len(available_numeric_sweep_features) - 1
print(f"Running {n_combos} numeric feature combinations: {available_numeric_sweep_features}")

combo_index = 0
for combo_size in range(1, len(available_numeric_sweep_features) + 1):
    for numeric_combo in combinations(available_numeric_sweep_features, combo_size):
        combo_index += 1
        selected_numeric_features = list(numeric_combo)
        feature_set_name = "combo_" + "_".join(selected_numeric_features)
        print(
            f"[{combo_index:03d}/{n_combos}] Running {feature_set_name}: "
            f"{selected_numeric_features}",
            flush=True,
        )

        combo_anime_df = pd.concat(
            [
                numeric_combo_base[selected_numeric_features],
                anime_genres_df,
                synopsis_svd_df,
            ],
            axis=1,
        ).dropna()

        combo_recommender = SimilarityRecommender()
        combo_recommender.create_anime_vectors(combo_anime_df)

        combo_evaluator = HitRateEvaluator(
            anime_df_scaled=combo_recommender.anime_df_scaled,
            anime_df=combo_anime_df,
            scores=user_scores,
            heldout_fraction=0.25,
        )

        _, combo_summary, _, _, combo_baseline_summary = (
            combo_evaluator.tune_bayesian_uncertainty(
                weights=[all_combo_uncertainty_weight],
                n_runs=all_combo_n_runs,
                top_ks=all_combo_top_ks,
                random_state=42,
                clip_predictions=all_combo_clip_predictions,
            )
        )

        if combo_baseline_summary is not None and not combo_baseline_summary.empty:
            combo_summary = combo_summary.merge(
                combo_baseline_summary,
                on="k",
                how="left",
            )

        for row in combo_summary.to_dict("records"):
            row["feature_set"] = feature_set_name
            row["numeric_features"] = ", ".join(selected_numeric_features)
            row["n_numeric_features"] = len(selected_numeric_features)
            row["n_runs"] = all_combo_n_runs
            row["clip_predictions"] = all_combo_clip_predictions
            row["svd_explained_variance"] = builder.svd_explained_variance
            all_combo_rows.append(row)

all_numeric_combo_results = pd.DataFrame(all_combo_rows)
all_numeric_combo_summary = all_numeric_combo_results.sort_values(
    ["k", "avg_precision_at_k", "avg_hit_rate"],
    ascending=[True, False, False],
)

metrics_path = (
    PROJECT_ROOT
    / "metrics"
    / "current_corpus_4793_anime"
    / f"all_numeric_combo_sweep_{all_combo_n_runs}run_precision_2026.csv"
)
metrics_path.parent.mkdir(parents=True, exist_ok=True)
all_numeric_combo_summary.to_csv(metrics_path, index=False)

print(f"Saved {metrics_path.relative_to(PROJECT_ROOT)}")
all_numeric_combo_summary.head(20)


Feature frame diagnostic: numeric_rows=4793, genre_rows=4933, synopsis_rows=4933, numeric_genre_overlap=4793, numeric_synopsis_overlap=4793, all_overlap=4793
User score counts:
1      1
3      2
4      8
5     10
6     57
7     58
8     49
9     59
10    18
Evaluator diagnostic: user_scores=262, candidate_rows=4793, rated_overlap=262, like_threshold=7.782, liked_overlap=126, heldout_likes_per_split=32
Running 255 numeric feature combinations: ['mean', 'popularity', 'statistics_num_list_users', 'watching', 'completed', 'on_hold', 'dropped', 'plan_to_watch']
[001/255] Running combo_mean: ['mean']
[002/255] Running combo_popularity: ['popularity']
[003/255] Running combo_statistics_num_list_users: ['statistics_num_list_users']
[004/255] Running combo_watching: ['watching']
[005/255] Running combo_completed: ['completed']
[006/255] Running combo_on_hold: ['on_hold']
[007/255] Running combo_dropped: ['dropped']
[008/255] Running combo_plan_to_watch: ['plan_to_watch']
[009/255] Running combo

,uncertainty_weight,k,avg_precision_at_k,std_precision_at_k,avg_hit_rate,std_hit_rate,avg_hits,baseline_avg_precision_at_k,baseline_std_precision_at_k,baseline_avg_hit_rate,baseline_std_hit_rate,baseline_avg_hits,feature_set,numeric_features,n_numeric_features,n_runs,clip_predictions,svd_explained_variance
74,14,5,0.60,0.183533,0.093750,0.028677,3.00,0.17,0.134164,0.026562,0.020963,0.85,combo_mean_popularity_watching,"mean, popularity, watching",3,20,False,0.406873
198,14,5,0.59,0.210013,0.092188,0.032814,2.95,0.17,0.134164,0.026562,0.020963,0.85,combo_mean_popularity_watching_dropped,"mean, popularity, watching, dropped",4,20,False,0.406873
80,14,5,0.56,0.211262,0.087500,0.033010,2.80,0.17,0.134164,0.026562,0.020963,0.85,combo_mean_popularity_dropped,"mean, popularity, dropped",3,20,False,0.406873
78,14,5,0.54,0.195744,0.084375,0.030585,2.70,0.17,0.134164,0.026562,0.020963,0.85,combo_mean_popularity_on_hold,"mean, popularity, on_hold",3,20,False,0.406873
196,14,5,0.54,0.195744,0.084375,0.030585,2.70,0.17,0.134164,0.026562,0.020963,0.85,combo_mean_popularity_watching_on_hold,"mean, popularity, watching, on_hold",4,20,False,0.406873
208,14,5,0.53,0.197617,0.082812,0.030878,2.65,0.17,0.134164,0.026562,0.020963,0.85,combo_mean_popularity_on_hold_dropped,"mean, popularity, on_hold, dropped",4,20,False,0.406873
20,14,5,0.52,0.219089,0.081250,0.034233,2.60,0.17,0.134164,0.026562,0.020963,0.85,combo_mean_watching,"mean, watching",2,20,False,0.406873
194,14,5,0.51,0.210013,0.079687,0.032814,2.55,0.17,0.134164,0.026562,0.020963,0.85,combo_mean_popularity_watching_completed,"mean, popularity, watching, completed",4,20,False,0.406873
346,14,5,0.51,0.210013,0.079687,0.032814,2.55,0.17,0.134164,0.026562,0.020963,0.85,combo_mean_popularity_watching_completed_dropped,"mean, popularity, watching, completed, dropped",5,20,False,0.406873
350,14,5,0.51,0.177408,0.079687,0.027720,2.55,0.17,0.134164,0.026562,0.020963,0.85,combo_mean_popularity_watching_on_hold_dropped,"mean, popularity, watching, on_hold, dropped",5,20,False,0.406873


## Results

This current-corpus sweep tested all `255` non-empty combinations of the available numeric fields with `genres + synopsis SVD` included for every combo. It used Bayesian Ridge with `uncertainty_weight=14`, `clip_predictions=False`, `n_runs=20`, and precision/hit-rate metrics only. The precision-only sweep is saved to `metrics/current_corpus_4793_anime/all_numeric_combo_sweep_20run_precision_2026.csv`.

| Read | Feature set | Numeric features | P@5 | Hits@5 | P@10 | Hits@10 | Notes |
| --- | --- | --- | ---: | ---: | ---: | ---: | --- |
| Best top-5 | `combo_mean_popularity_watching` | mean, popularity, watching | 0.600 | 3.00 | 0.380 | 3.80 | Best short-list precision; matches the previous-cache selected feature set. |
| Close top-5 | `combo_mean_popularity_watching_dropped` | mean, popularity, watching, dropped | 0.590 | 2.95 | 0.370 | 3.70 | Very close, but adds an extra feature without beating the simpler winner. |
| Third top-5 | `combo_mean_popularity_dropped` | mean, popularity, dropped | 0.560 | 2.80 | 0.360 | 3.60 | Strong, but weaker than keeping watching. |
| Best top-10 | `combo_popularity_watching_on_hold_dropped` | popularity, watching, on_hold, dropped | n/a | n/a | 0.415 | 4.15 | Best P@10 in this 20-run sweep, but not the app's main short-list objective. |
| Simple candidate | `combo_mean_watching` | mean, watching | 0.520 | 2.60 | 0.375 | 3.75 | Competitive, but lower P@5 than mean + popularity + watching. |

Decision: stick with **mean + popularity + watching + genres + synopsis SVD**. It wins Precision@5 on the current cache and it also matches the best feature set from the previous-cache experiments, so I am not doing extra confirmation runs for feature selection right now. The small P@10-only winner is noted, but top-5 precision is the main recommendation-list target and the simpler previous winner remains the most consistent choice.
